# TL;DR: Unbekannte Teilnehmer und uneindeutige Ordner verursachen die meisten Fehler

Der zeitliche Backtest von **785 Mails aus neuen Threads** erreicht mit Snowflake Arctic Embed 2 plus Teilnehmerhistorie **68.3% Top-1** und **82.5% Top-3**. Von 249 falschen Erstvorschlägen liegen 110 im Eltern-, Kind- oder direkten Geschwisterordner. Der wichtigste Risikofaktor ist fehlende Teilnehmerhistorie: nur 44,8 % Top-1.

Die Auswertung speichert weder Mailtexte noch reale Betreffzeilen, Adressen oder Ordnerpfade.

## Context & Methods

- Ältere 80 % je geeignetem Ordner dienen als Training, jüngere 20 % als Test.
- Die Hauptkohorte enthält nur Testmails, deren Thread im Training nicht vorkommt.
- Pro Ordner wird ein Zentroid aus Snowflake-Arctic-Embed-2-Vektoren gebildet.
- Eine ausreichend eindeutige Historie aus `From`, `To` und `Cc` darf die semantische Rangfolge umsortieren (Gate 0,5).
- Der komplette lokal verfügbare Nachrichtentext wird verwendet, aber auf 6.000 Zeichen je Mail begrenzt.
- Segmentvergleiche sind diagnostisch und nicht kausal; einige Merkmale überlappen.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

base = Path.cwd()
if not (base / "error_diagnostics.json").exists():
    base = Path("analysis/mail_error_diagnostics_2026-08-25")
result = json.loads((base / "error_diagnostics.json").read_text(encoding="utf-8"))

def segment_frame(segment_name):
    rows = []
    for label, values in result["segments"][segment_name].items():
        rows.append({"Segment": label, **values})
    frame = pd.DataFrame(rows)
    return frame.rename(columns={
        "n": "Mails", "top1": "Top-1", "top3": "Top-3",
        "top1_errors": "Top-1-Fehler", "top3_misses": "Top-3-Fehler",
    })

def format_rates(frame):
    return frame.style.format({"Top-1": "{:.1%}", "Top-3": "{:.1%}"})

## Data

In [2]:
coverage = pd.DataFrame([{
    "Indexierte Mails": result["indexed_messages"],
    "Mit lokalem Volltext": result["usable_full_text_messages"],
    "Volltextabdeckung": result["usable_full_text_messages"] / result["indexed_messages"],
    "Training": result["train_messages"],
    "Test": result["test_messages"],
    "Neue Threads": result["new_thread_test_messages"],
    "Geeignete Ordner": result["split"]["eligible_folders"],
}])
display(coverage.style.format({"Volltextabdeckung": "{:.1%}"}))

,Indexierte Mails,Mit lokalem Volltext,Volltextabdeckung,Training,Test,Neue Threads,Geeignete Ordner
0,17087,4503,26.4%,3480,876,785,107


In [3]:
# Reproduzierbare Konsistenzprüfungen
assert result["train_messages"] + result["test_messages"] + result["split"]["excluded_sparse_messages"] == result["usable_full_text_messages"]
assert result["overall"]["n"] == result["new_thread_test_messages"]
assert result["overall"]["top1_errors"] == round(result["overall"]["n"] * (1 - result["overall"]["top1"]))
assert result["overall"]["top3_misses"] == round(result["overall"]["n"] * (1 - result["overall"]["top3"]))
for segment in ["attachment", "direction", "body_length", "folder_train_size", "semantic_margin", "participant_history", "target_kind", "folder_depth"]:
    assert sum(row["n"] for row in result["segments"][segment].values()) == result["overall"]["n"], segment
assert sum(row["n"] for row in result["segments"]["hierarchy_relation"].values()) == result["overall"]["top1_errors"]
assert result["overall"]["top1"] <= result["overall"]["top3"]
print("Alle Konsistenzprüfungen bestanden.")

Alle Konsistenzprüfungen bestanden.


## Results

### 1. Teilnehmerhistorie ist der stärkste Trennfaktor

Wenn das Gate greift, ist der erste Vorschlag fast doppelt so häufig richtig wie bei völlig unbekannten Teilnehmern. Gegenüber reiner Semantik verbessert das Teilnehmer-Gate 222 Mails und verschlechtert 19.

In [4]:
display(format_rates(segment_frame("participant_history")))

,Segment,Mails,Top-1,Top-3,Top-1-Fehler,Top-3-Fehler
0,Gate greift,532,78.8%,89.7%,113,55
1,Keine Historie,125,44.8%,59.2%,69,51
2,Zu unsicher,128,47.7%,75.8%,67,31


In [5]:
gate = result["participant_gate"]
pd.DataFrame([{
    "Durch Gate verbessert": gate["helped_top1"],
    "Durch Gate verschlechtert": gate["harmed_top1"],
    "Top-1 nur Semantik": gate["semantic_only"]["top1"],
    "Top-1 kombiniert": result["overall"]["top1"],
}]).style.format({"Top-1 nur Semantik": "{:.1%}", "Top-1 kombiniert": "{:.1%}"})

,Durch Gate verbessert,Durch Gate verschlechtert,Top-1 nur Semantik,Top-1 kombiniert
0,222,19,42.4%,68.3%


### 2. Lange und weitergeleitete Inhalte sind schwieriger

Mittellange Volltexte funktionieren am besten. Oberhalb der 6.000-Zeichen-Grenze fällt Top-1 auf 52,0 %. Weiterleitungen erreichen 58,2 %. Anhänge allein sind kein starker Top-3-Nachteil; problematisch werden sie vor allem bei wenig Begleittext.

In [6]:
display(format_rates(segment_frame("body_length")))

,Segment,Mails,Top-1,Top-3,Top-1-Fehler,Top-3-Fehler
0,1.001–4.000 Zeichen,339,71.7%,86.4%,96,46
1,251–1.000 Zeichen,300,66.7%,81.7%,100,55
2,4.001–6.000 Zeichen,40,67.5%,77.5%,13,9
3,>6.000 Zeichen (gekürzt),25,52.0%,68.0%,12,8
4,≤250 Zeichen,81,65.4%,76.5%,28,19


In [7]:
flags = segment_frame("overlapping_flags")
focus = flags[flags["Segment"].isin([
    "Weiterleitungsbetreff", "Anhang mit wenig Begleittext",
    "Überwiegend Zitat/Signatur", "Antwortbetreff",
])]
display(format_rates(focus))

,Segment,Mails,Top-1,Top-3,Top-1-Fehler,Top-3-Fehler
0,Anhang mit wenig Begleittext,118,62.7%,78.0%,44,26
1,Antwortbetreff,237,76.8%,90.3%,55,23
7,Weiterleitungsbetreff,55,58.2%,70.9%,23,16
8,Überwiegend Zitat/Signatur,142,69.0%,78.2%,44,31


### 3. Viele Fehler sind hierarchische Nahfehler

110 von 249 falschen Erstvorschlägen (44,2 %) liegen im Eltern-, Kind- oder direkten Geschwisterordner. Nur 19 Fehler wechseln den Hauptzweig. Der gemeinsame Archiv-Wurzelordner macht „gleicher Hauptzweig“ allerdings zu einer groben Kennzahl.

In [8]:
hierarchy = segment_frame("hierarchy_relation")
hierarchy["Anteil aller Top-1-Fehler"] = hierarchy["Mails"] / result["overall"]["top1_errors"]
display(hierarchy[["Segment", "Mails", "Anteil aller Top-1-Fehler", "Top-3"]].style.format({
    "Anteil aller Top-1-Fehler": "{:.1%}", "Top-3": "{:.1%}",
}))

,Segment,Mails,Anteil aller Top-1-Fehler,Top-3
0,Vorschlag ist ein Unterordner des tatsächlichen Ordners,35,14.1%,48.6%
1,Vorschlag ist ein übergeordneter Ordner,11,4.4%,45.5%
2,anderer Hauptzweig,19,7.6%,36.8%
3,benachbarter Ordner mit gleichem Elternordner,64,25.7%,59.4%
4,gleicher Hauptzweig,120,48.2%,37.5%


In [9]:
display(format_rates(segment_frame("folder_depth")))

,Segment,Mails,Top-1,Top-3,Top-1-Fehler,Top-3-Fehler
0,1–2 Ebenen,21,33.3%,52.4%,14,10
1,3 Ebenen,113,61.1%,74.3%,44,29
2,4 Ebenen,194,63.4%,78.3%,71,42
3,≥5 Ebenen,457,73.7%,87.8%,120,56


### 4. Kleine Ordner sind nicht pauschal das Problem

Die kleinste auswertbare Trainingsgruppe schneidet in diesem Snapshot sogar gut ab. Das ist kein Beleg, dass wenig Daten helfen; die Kohorte umfasst nur 34 Testmails und enthält vermutlich besonders eindeutige Teilnehmer oder Themen. Historische systemähnliche Altordner sind dagegen deutlich schlechter erkannt und erzeugen vermeidbares Labelrauschen.

In [10]:
display(format_rates(segment_frame("folder_train_size")))

,Segment,Mails,Top-1,Top-3,Top-1-Fehler,Top-3-Fehler
0,11–25 Trainingsmails,113,74.3%,81.4%,29,21
1,26–100 Trainingsmails,444,65.8%,82.4%,152,78
2,6–10 Trainingsmails,34,76.5%,88.2%,8,4
3,>100 Trainingsmails,194,69.1%,82.5%,60,34


In [11]:
display(format_rates(segment_frame("target_kind")))

,Segment,Mails,Top-1,Top-3,Top-1-Fehler,Top-3-Fehler
0,Fachlicher Archivordner,774,68.7%,83.2%,242,130
1,Systemähnlicher Altordner,11,36.4%,36.4%,7,7


## Takeaways

1. **Teilnehmerhistorie produktiv nutzen:** eigene Adressen explizit konfigurieren und `From`, `To`, `Cc` gemeinsam auswerten.
2. **Unsicherheit sichtbar machen:** ohne Teilnehmerhistorie und bei kleinem Embedding-Abstand lieber mehrere Vorschläge oder „unsicher“ anzeigen; niemals automatisch verschieben.
3. **Ordnerbaum berücksichtigen:** zuerst Objekt/Hauptordner, danach Unterordner bewerten; direkte Nachbarn getrennt nachranken.
4. **Anhänge erschließen:** Dateiname, MIME-Typ und später extrahierten Anhangtext als zusätzliche Merkmale testen.
5. **Weiterleitungen bereinigen:** neuen Begleittext, weitergeleitete Header und eingebettete Ursprungsnachricht getrennt gewichten.
6. **Pseudo-Systemordner prüfen:** alte Entwurfs-, Papierkorb- oder Posteingangsordner nur als Ziele zulassen, wenn sie fachlich wirklich gewollt sind.

### Grenzen

Nur 4.503 von 17.087 indexierten Mails hatten im lokalen Thunderbird-Cache verwertbaren Volltext. Die Segmentergebnisse können deshalb durch den Cachebestand verzerrt sein. Der Lauf misst historische Ablageentscheidungen, die teilweise selbst inkonsistent oder mehrdeutig sein können. Das derzeit konfigurierte Archivkonto hat Embeddings noch deaktiviert; die Ergebnisse beschreiben den getesteten Zielansatz, nicht den momentan laufenden Produktivklassifikator.